# 09 — Xuất ảnh chú thích 15 ca phân loại sai (YOLOv11m + EfficientNet-B2)

**Mục đích.** Sinh ảnh chú thích cho đúng mười lăm ca sai của cấu hình đề xuất, để thay
cho hai ảnh đang mượn tạm từ cấu hình tham chiếu YOLOv11m + ResNet50 ở mục 5.5.2 của
báo cáo tổng kết SV2026-08.

**Điểm quan trọng: notebook này KHÔNG chạy lại mô hình.** Tệp
`outputs/pipeline_evaluation/yolov11_m__efficientnet_b2/error_cases.csv` đã lưu sẵn toạ độ
từng khung (`xyxy`), lớp và độ tin cậy YOLO gán cho khung, lớp và độ tin cậy EfficientNet-B2
gán cho vùng cắt tương ứng. Vì vậy chỉ cần vẽ lại — không cần GPU, không cần `ultralytics`,
không cần nạp checkpoint. Chạy hết khoảng một phút.

**Lý do phải chạy trên Colab:** thư mục ảnh kiểm tra gốc (291 ảnh) nằm trên Google Drive
chứ không có trong repo.

**Cần chuẩn bị**
1. Google Drive đã gắn, có thư mục `Papaya_Leaf_Disease_Classification/prepared_data_v1/yolo_dataset/images/test/`.
2. Tệp `error_cases.csv` — notebook tự tìm trên Drive, không thấy thì sẽ hỏi tải lên từ máy.

**Kết quả**
| Tệp | Dùng vào đâu |
|---|---|
| `ca01…ca15_*.png` | 15 ảnh chú thích đầy đủ |
| `hinh_5_5_2_1_*.png`, `hinh_5_5_2_2_*.png` | thay trực tiếp hai ảnh ở mục 5.5.2 |
| `bang_anh_15_ca_sai.png` | lưới 3×5 cả 15 ca, dùng cho phụ lục |

Chạy tuần tự từ trên xuống. Cell cuối tự nén và tải về máy, đồng thời lưu một bản lên Drive.


In [ ]:
# ============================================================================
# 1. Gắn Google Drive và khai báo đường dẫn
# ============================================================================
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Papaya_Leaf_Disease_Classification"

# Thư mục chứa 291 ảnh kiểm tra gốc (đúng đường dẫn đã ghi trong error_cases.csv).
IMAGES_DIR = f"{DRIVE_ROOT}/prepared_data_v1/yolo_dataset/images/test"

# Nơi lưu ảnh chú thích sinh ra.
OUT_DIR    = "/content/annotated_errors_effb2"

import os
print("Thư mục ảnh gốc tồn tại:", os.path.isdir(IMAGES_DIR))
if os.path.isdir(IMAGES_DIR):
    print("Số tệp trong thư mục:", len(os.listdir(IMAGES_DIR)))
else:
    print("!! Hãy sửa lại IMAGES_DIR cho khớp với cấu trúc Drive của bạn.")


In [ ]:
# ============================================================================
# 2. Nạp error_cases.csv của cấu hình YOLOv11m + EfficientNet-B2
# ============================================================================
# Tệp này nằm trong repo tại:
#   outputs/pipeline_evaluation/yolov11_m__efficientnet_b2/error_cases.csv
# Notebook thử tìm trên Drive trước; không thấy thì hỏi tải lên thủ công.
import os
import pandas as pd

CANDIDATES = [
    f"{DRIVE_ROOT}/outputs/pipeline_evaluation/yolov11_m__efficientnet_b2/error_cases.csv",
    f"{DRIVE_ROOT}/pipeline_evaluation/yolov11_m__efficientnet_b2/error_cases.csv",
    "/content/error_cases.csv",
]

csv_path = next((p for p in CANDIDATES if os.path.isfile(p)), None)

if csv_path is None:
    print("Không tìm thấy error_cases.csv. Hãy chọn tệp từ máy của bạn:")
    from google.colab import files
    up = files.upload()
    csv_path = "/content/" + next(iter(up))

df = pd.read_csv(csv_path)
print("Nguồn:", csv_path)
print("Số ca sai:", len(df))
assert len(df) == 15, f"Cần đúng 15 ca sai như Bảng 25, đang có {len(df)}"
df[["true_class", "pred_class", "reason", "num_boxes", "conf_used"]]


In [ ]:
# ============================================================================
# 3. Bộ công cụ vẽ chú thích  (thuần PIL — KHÔNG nạp mô hình, KHÔNG cần GPU)
# ============================================================================
# Mọi thông tin cần thiết đã nằm sẵn trong error_cases.csv: toạ độ từng khung
# (xyxy), lớp và độ tin cậy mà YOLO gán cho khung, lớp và độ tin cậy mà CNN gán
# cho vùng cắt tương ứng. Vì vậy bước này chỉ là vẽ lại, chạy vài giây trên CPU.
import os, json
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

TARGET_W  = 1100          # phóng ảnh lên cho chữ chú thích đủ nét khi in báo cáo
C_OK      = (22, 163, 74)     # xanh lá – vùng cắt được phân loại đúng lớp thực tế
C_BAD     = (220, 38, 38)     # đỏ      – vùng cắt được phân loại sai
C_FULL    = (217, 119, 6)     # cam     – trường hợp rơi về nhánh phân loại toàn ảnh
C_HEADER  = (31, 41, 55)
C_WHITE   = (255, 255, 255)

VN = {  # tên lớp hiển thị bằng tiếng Việt cho phần tiêu đề
    "Anthracnose":   "Thán thư",
    "BacterialSpot": "Đốm vi khuẩn",
    "Curl":          "Xoăn lá",
    "Healthy":       "Lá khỏe",
    "RingSpot":      "Đốm vòng",
}
REASON_VN = {
    "yolo_boxes_cnn_weighted_vote":
        "Bỏ phiếu có trọng số trên các khung YOLO",
    "no_yolo_box_after_conf_retry":
        "Không còn khung nào sau khi đã hạ ngưỡng tin cậy",
    "no_yolo_box_full_cnn_healthy_or_low_conf":
        "Không có khung → dùng nhánh phân loại toàn ảnh",
}

# Phông phải phủ đủ dấu tiếng Việt. DejaVu có sẵn trên Colab và đạt yêu cầu;
# nếu vì lý do nào đó không có, tải Roboto về làm dự phòng.
FONT_PATHS = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
    "/content/Roboto-Bold.ttf",
]

if not any(os.path.isfile(p) for p in FONT_PATHS):
    !wget -q -O /content/Roboto-Bold.ttf https://github.com/google/fonts/raw/main/apache/roboto/static/Roboto-Bold.ttf

def _font(size):
    for fp in FONT_PATHS:
        try:
            return ImageFont.truetype(fp, size)
        except OSError:
            continue
    print("!! Không tìm thấy phông TrueType — chữ tiếng Việt có thể mất dấu.")
    return ImageFont.load_default()

# Kiểm tra nhanh: nếu phông thiếu glyph, chuỗi dưới sẽ hiện ra các ô vuông rỗng.
_probe = Image.new("RGB", (620, 44), (31, 41, 55))
ImageDraw.Draw(_probe).text((8, 8), "Kiểm tra dấu: ệ ữ ẩ ỗ ợ Đ — phải đọc được",
                            font=_font(22), fill=(255, 255, 255))
display(_probe)

def _num(x, nd=2):
    """Số thập phân theo quy ước tiếng Việt (dấu phẩy)."""
    return f"{x:.{nd}f}".replace(".", ",")

def _fit(draw, text, max_w, start=25, floor=14):
    """Chọn cỡ chữ lớn nhất mà dòng vẫn nằm gọn trong bề ngang cho trước."""
    size = start
    while size > floor and draw.textlength(text, font=_font(size)) > max_w:
        size -= 1
    return _font(size)

def _chip(draw, xy, text, font, fill):
    """Nhãn nền đặc, chữ trắng — đọc được trên mọi nền ảnh."""
    x, y = xy
    l, t, r, b = draw.textbbox((0, 0), text, font=font)
    w, h = r, b
    l = t = 0
    pad = 4
    draw.rectangle([x, y, x + w + 2 * pad, y + h + 2 * pad], fill=fill)
    draw.text((x + pad - l, y + pad - t), text, font=font, fill=C_WHITE)
    return w + 2 * pad, h + 2 * pad

def annotate(row, idx):
    """Trả về ảnh đã chú thích cho một dòng của error_cases.csv."""
    src = os.path.join(IMAGES_DIR, os.path.basename(row["image_path"]))
    img = Image.open(src).convert("RGB")
    s = TARGET_W / img.width
    img = img.resize((TARGET_W, round(img.height * s)), Image.LANCZOS)

    f_box = _font(19)

    # -------- dải tiêu đề phía trên ảnh --------
    head_h = 92
    canvas = Image.new("RGB", (img.width, img.height + head_h), C_HEADER)
    canvas.paste(img, (0, head_h))
    d = ImageDraw.Draw(canvas)

    gt, pr = row["true_class"], row["pred_class"]
    head = (f"Ca {idx:02d}  •  Thực tế: {VN.get(gt, gt)} ({gt})"
            f"   →   Dự đoán: {VN.get(pr, pr)} ({pr})")

    sub = (f"{REASON_VN.get(row['reason'], row['reason'])}  |  "
           f"số khung: {int(row['num_boxes'])}  |  "
           f"ngưỡng dùng: {_num(row['conf_used'])}")
    if isinstance(row.get("full_cnn_class"), str) and row["full_cnn_class"]:
        sub += (f"  |  toàn ảnh: {row['full_cnn_class']} "
                f"{_num(float(row['full_cnn_conf']))}")

    # Hai dòng tiêu đề tự co lại nếu dài hơn bề ngang ảnh, tránh bị cắt cụt.
    avail = img.width - 32
    d.text((16, 12), head, font=_fit(d, head, avail, 25, 16), fill=C_WHITE)
    d.text((16, 52), sub, font=_fit(d, sub, avail, 20, 13), fill=(203, 213, 225))

    # -------- các khung phát hiện --------
    crops = json.loads(row["crop_predictions_json"]) if isinstance(
        row["crop_predictions_json"], str) else []

    if not crops:
        # Nhánh dự phòng: không có khung nào, đóng khung cả ảnh cho rõ ý.
        d.rectangle([3, head_h + 3, img.width - 4, canvas.height - 4],
                    outline=C_FULL, width=4)
        _chip(d, (12, head_h + 12), "Không có khung — phân loại toàn ảnh",
              f_box, C_FULL)
    else:
        for c in crops:
            x1, y1, x2, y2 = [v * s for v in c["xyxy"]]
            y1 += head_h; y2 += head_h
            # Tô theo việc vùng cắt bỏ phiếu đúng hay sai so với lớp thực tế của ảnh:
            # các khung đỏ chính là những phiếu đã kéo kết quả tổng hợp lệch đi.
            color = C_OK if c["cnn_class"] == gt else C_BAD
            d.rectangle([x1, y1, x2, y2], outline=color, width=3)
            label = f"{c['cnn_class']} {_num(c['cnn_conf'])}"
            cw, ch = d.textbbox((0, 0), label, font=f_box)[2:]
            cw, ch = cw + 8, ch + 8
            # Ưu tiên đặt nhãn phía trên khung; nếu không đủ chỗ (khung sát mép trên
            # hoặc phủ gần kín ảnh) thì đặt lọt vào bên trong, không bao giờ để tràn ra
            # ngoài ảnh vì phần tràn sẽ bị cắt mất.
            if y1 - ch - 2 > head_h:
                ly = y1 - ch - 2
            elif y2 + ch + 2 < canvas.height:
                ly = y2 + 2
            else:
                ly = y1 + 2
            lx = min(max(2, x1), canvas.width - cw - 2)
            _chip(d, (lx, ly), label, f_box, color)

    # -------- chú giải màu dưới đáy --------
    leg_h = 40
    out = Image.new("RGB", (canvas.width, canvas.height + leg_h), C_HEADER)
    out.paste(canvas, (0, 0))
    d2 = ImageDraw.Draw(out)
    items = [(C_OK,   "vùng cắt phân loại đúng lớp thực tế"),
             (C_BAD,  "vùng cắt phân loại sai"),
             (C_FULL, "không có khung — dùng nhánh toàn ảnh")]
    f_leg = _fit(d2, "".join(t for _, t in items) + "      ", out.width - 140, 20, 12)
    x = 16
    for color, text in items:
        d2.rectangle([x, canvas.height + 13, x + 26, canvas.height + 29], fill=color)
        d2.text((x + 34, canvas.height + 11), text, font=f_leg, fill=(203, 213, 225))
        x += 40 + d2.textlength(text, font=f_leg)
    return out

print("Đã định nghĩa hàm annotate().")


In [ ]:
# ============================================================================
# 4. Sinh 15 ảnh chú thích
# ============================================================================
import os, shutil
from IPython.display import display

shutil.rmtree(OUT_DIR, ignore_errors=True)
os.makedirs(OUT_DIR, exist_ok=True)

paths = []
for i, row in df.iterrows():
    idx = i + 1
    try:
        im = annotate(row, idx)
    except FileNotFoundError:
        print(f"  !! Thiếu ảnh gốc: {os.path.basename(row['image_path'])}")
        continue
    name = f"ca{idx:02d}_gt-{row['true_class']}_pred-{row['pred_class']}.png"
    fp = os.path.join(OUT_DIR, name)
    im.save(fp, quality=95)
    paths.append(fp)
    print(f"  [{idx:02d}/15] {name}")

print(f"\nĐã sinh {len(paths)} ảnh trong {OUT_DIR}")
display(Image.open(paths[0]).resize((760, int(760 * Image.open(paths[0]).height
                                              / Image.open(paths[0]).width))))


In [ ]:
# ============================================================================
# 5. Hai hình dùng trực tiếp cho báo cáo
# ============================================================================
# (a) Hình 2 ảnh thay cho hình hiện tại ở mục 5.5.2 — chọn đúng hai ca mà phần
#     lời văn đang mô tả: Healthy→BacterialSpot và BacterialSpot→Curl.
# (b) Bảng ảnh 15 ca (lưới 3x5) để đưa vào phần phụ lục.

import os

def pick(gt, pred):
    m = df[(df.true_class == gt) & (df.pred_class == pred)]
    if m.empty:
        return None
    i = m.index[0]
    return annotate(df.loc[i], i + 1)

PAIR = [("Healthy", "BacterialSpot"), ("BacterialSpot", "Curl")]
imgs = [im for im in (pick(*p) for p in PAIR) if im is not None]

if len(imgs) == 2:
    for k, (im, (gt, pr)) in enumerate(zip(imgs, PAIR), 1):
        fp = os.path.join(OUT_DIR, f"hinh_5_5_2_{k}_gt-{gt}_pred-{pr}.png")
        im.save(fp)
        print("Đã lưu:", os.path.basename(fp))

# ---- bảng ảnh 15 ca ----
COLS, ROWS, THUMB_W, GAP = 3, 5, 620, 14
thumbs = []
for i in range(len(df)):
    try:
        im = annotate(df.loc[i], i + 1)
    except FileNotFoundError:
        continue
    thumbs.append(im.resize((THUMB_W, round(THUMB_W * im.height / im.width)),
                            Image.LANCZOS))

if thumbs:
    cell_h = max(t.height for t in thumbs)
    sheet = Image.new("RGB",
                      (COLS * THUMB_W + (COLS + 1) * GAP,
                       ROWS * cell_h + (ROWS + 1) * GAP),
                      (255, 255, 255))
    for k, t in enumerate(thumbs):
        r, c = divmod(k, COLS)
        sheet.paste(t, (GAP + c * (THUMB_W + GAP), GAP + r * (cell_h + GAP)))
    sheet_fp = os.path.join(OUT_DIR, "bang_anh_15_ca_sai.png")
    sheet.save(sheet_fp)
    print("Đã lưu:", os.path.basename(sheet_fp), sheet.size)
    display(sheet.resize((820, round(820 * sheet.height / sheet.width))))


In [ ]:
# ============================================================================
# 6. Đóng gói và tải về máy
# ============================================================================
# Sau khi tải về, giải nén vào đúng thư mục trong repo:
#   outputs/pipeline_evaluation/yolov11_m__efficientnet_b2/annotated_images/
import shutil
from google.colab import files

zip_base = "/content/annotated_errors_effb2"
shutil.make_archive(zip_base, "zip", OUT_DIR)
print("Kích thước:", round(os.path.getsize(zip_base + ".zip") / 1e6, 2), "MB")

# Chép thêm một bản lên Drive để khỏi phải làm lại nếu phiên Colab bị ngắt.
drive_out = f"{DRIVE_ROOT}/outputs/pipeline_evaluation/yolov11_m__efficientnet_b2/annotated_images"
os.makedirs(drive_out, exist_ok=True)
for f in os.listdir(OUT_DIR):
    shutil.copy2(os.path.join(OUT_DIR, f), drive_out)
print("Đã chép lên Drive:", drive_out)

files.download(zip_base + ".zip")


## Sau khi chạy xong

1. Giải nén tệp `annotated_errors_effb2.zip` vừa tải về vào
   `outputs/pipeline_evaluation/yolov11_m__efficientnet_b2/annotated_images/` trong repo.
2. Trong `BaoCaoTongKet_SV2026-08_FULL.docx`, tới mục **5.5.2. Minh họa trực quan các ca sai**:
   thay hai ảnh hiện tại bằng `hinh_5_5_2_1_gt-Healthy_pred-BacterialSpot.png` và
   `hinh_5_5_2_2_gt-BacterialSpot_pred-Curl.png`. Hai ca này được chọn đúng bằng hai ca mà
   chú thích hình hiện tại đang mô tả, nên phần lời văn và chú thích giữ nguyên, không phải sửa.
3. Xoá đoạn `[CẦN BỔ SUNG]` ngay dưới hình đó.
4. Nếu muốn đưa cả 15 ca vào phụ lục, chèn `bang_anh_15_ca_sai.png` với chú thích:
   *"Toàn bộ mười lăm ca phân loại sai của cấu hình YOLOv11m + EfficientNet-B2 trên tập kiểm tra"*.

**Quy ước màu trong ảnh chú thích**
- Xanh lá — vùng cắt được EfficientNet-B2 phân loại **đúng** lớp thực tế của ảnh.
- Đỏ — vùng cắt được phân loại **sai**; đây chính là các phiếu đã kéo kết quả bỏ phiếu lệch đi.
- Cam — khung phủ gần hết ảnh, hoặc trường hợp không có khung nào và hệ thống rơi về nhánh
  phân loại toàn ảnh.
